# FrameNet と PropBank で述語の意味フレームと意味役割を調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_semantics/framenet_propbank.ipynb)

NLP クックブック 19-02 ／ 意味解析

動詞 `buy` がどんな参加者（買い手、商品、売り手…）を取るのかを、言語資源から引きたい。意味役割付与のラベル体系を理解したい。

In [1]:
import nltk
nltk.download("framenet_v17", quiet=True)
nltk.download("propbank", quiet=True)
from nltk.corpus import framenet as fn

frame = fn.frame("Commerce_buy")
print(frame.name, "-", frame.definition[:160])
print("中核的なフレーム要素:", [name for name, fe in frame.FE.items() if fe.coreType == "Core"])
print("周辺的なフレーム要素:", [name for name, fe in frame.FE.items() if fe.coreType != "Core"][:8])
print("このフレームを喚起する語:", sorted(frame.lexUnit)[:8])

Commerce_buy - These are words describing a basic commercial transaction involving a Buyer and a Seller exchanging Money and Goods, taking the perspective of the Buyer.  The w
中核的なフレーム要素: ['Buyer', 'Goods']
周辺的なフレーム要素: ['Means', 'Money', 'Rate', 'Seller', 'Unit', 'Place', 'Purpose', 'Explanation']
このフレームを喚起する語: ['buy.v', 'buyer.n', 'client.n', 'purchase [act].n', 'purchase.v', 'purchaser.n']


In [2]:
# 単語からフレームを探す / フレーム間の関係をたどる
print([f.name for f in fn.frames_by_lemma(r"(?i)\bsell\b")])
print([f"{r.superFrameName} → {r.subFrameName} ({r.type.name})" for r in frame.frameRelations][:4])

# 注釈付きの例文: どの部分がどのフレーム要素か
lu = [l for l in fn.lus(r"(?i)^buy\.v$") if l.frame.name == "Commerce_buy"][0]
for ex in lu.exemplars[:3]:
    print(ex.text.strip())
    print("   ", [(ex.text[s:e], role) for s, e, role in ex.FE[0]])

['Commerce_sell', 'Short_selling']
['Commerce_buy → Renting (Inheritance)', 'Getting → Commerce_buy (Inheritance)', 'Commerce_buy → Importing (Using)', 'Commerce_buy → Shopping (Using)']
Four years ago I bought an old Harmony Sovereign acoustic guitar for £20 from an absolute prat .
    [('I', 'Buyer'), ('an old Harmony Sovereign acoustic guitar', 'Goods'), ('for £20', 'Money'), ('from an absolute prat', 'Seller')]
Heston , who had director , script and cast approval , bought Peckinpah extra filming time by returning his salary to the studio , convinced they would n't accept it .
    [('Heston', 'Buyer'), ('Peckinpah', 'Recipient'), ('extra filming time', 'Goods'), ('by returning his salary to the studio', 'Means')]
On other occasions , borrowing may be the only way you will ever be able to afford to buy something expensive like a house .
    [('you', 'Buyer'), ('something expensive', 'Goods')]


In [3]:
# PropBank: 動詞の語義 (roleset) ごとに、番号付きの項 (Arg0, Arg1, ...) を定義する
from nltk.corpus import propbank

for roleset_id in ["buy.01", "sell.01", "give.01"]:
    rs = propbank.roleset(roleset_id)
    print(roleset_id, rs.attrib["name"], {f"Arg{r.attrib['n']}": r.attrib["descr"] for r in rs.findall("roles/role")})

buy.01 purchase {'Arg0': 'buyer', 'Arg1': 'thing bought', 'Arg2': 'seller', 'Arg3': 'price paid', 'Arg4': 'benefactive'}
sell.01 commerce: seller {'Arg0': 'Seller', 'Arg1': 'Thing Sold', 'Arg2': 'Buyer', 'Arg3': 'Price Paid', 'Arg4': 'Benefactive'}
give.01 transfer {'Arg0': 'giver', 'Arg1': 'thing given', 'Arg2': 'entity given to'}


- FrameNet は「場面」（フレーム）を単位に、場面ごとの役割（Buyer、Goods、Seller）を定義する。`buy` と `sell` は同じ商取引の場面を別の視点から述べたものとして関連付けられる。
- PropBank は動詞ごとに Arg0（動作主的な項）、Arg1（対象的な項）… と番号を振る。ラベルが少なく一貫しているので、意味役割付与の学習データとして広く使われる。
- 日本語には、日本語フレームネット、京都大学格フレーム、NPCMJ の PropBank 風アノテーションなどがある。